# Experiment 2

Analysis run with HSSM 0.2.4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import gaussian_kde
from IPython.display import display

## Data cleaning & filtering (can be skipped)

In [ ]:
df = pd.read_csv('df_study2_raw_noID.csv')
df = df.drop(df.columns[0], axis=1)

In [ ]:
# Internal node IDs holding the prior belief scores
node_IDs_priors = ['0.0-5.0', '0.0-6.0', '0.0-7.0', '0.0-8.0', '0.0-9.0']

# A = Climate Change, B = Vaccination, C = GMO, D = 5G, E = Voter Fraud
prior_letters = ['A', 'B', 'C', 'D', 'E']

In [ ]:
# Internal node IDs holding the source credibility scores
node_IDs_sources = ['0.0-11.0', '0.0-12.0', '0.0-13.0', '0.0-14.0', '0.0-15.0', '0.0-16.0', '0.0-17.0']

news_sources = {
    '0.0-11.0': 'CNN',
    '0.0-12.0': 'Fox',
    '0.0-13.0': 'NBC',
    '0.0-14.0': 'ABC',
    '0.0-15.0': 'CBS',
    '0.0-16.0': 'USAtoday',
    '0.0-17.0': 'NYT'
}

In [ ]:
# Remove participants that did not finish the experiment
filtered_df = df[df['internal_node_id'] == '0.0-227.0']
unique_participants = filtered_df['participant'].unique()
df = df[df['participant'].isin(unique_participants)]

In [ ]:
# Priors, source credibilities and political affiliation, per participant
source_credibilities = pd.DataFrame(columns=['participant', 'CNN', 'Fox', 'NBC', 'ABC', 'CBS', 'USAtoday', 'NYT', 'partisanship'])

news_sources = dict(zip(node_IDs_sources, ['CNN', 'Fox', 'NBC', 'ABC', 'CBS', 'USAtoday', 'NYT']))

for i in range(len(unique_participants)):
    current_participant = unique_participants[i]

    priors = pd.to_numeric(
        df[(df['internal_node_id'].isin(node_IDs_priors)) &
           (df['participant'] == unique_participants[i])]['response']
        .reset_index(drop=True))

    for j in range(len(prior_letters)):
        if j < len(priors):
            df.loc[df['participant'] == unique_participants[i],
                  f'prior_{prior_letters[j]}'] = priors[j]

    participant_sources = {'participant': current_participant}

    for node_id in node_IDs_sources:
        credibility = df[(df['participant'] == current_participant) &
                        (df['internal_node_id'] == node_id)]['response'].iloc[0]
        participant_sources[news_sources[node_id]] = credibility

    source_credibilities.loc[i] = participant_sources

    partisanship_data = df[(df['participant'] == unique_participants[i]) &
                          (df['internal_node_id'] == '0.0-227.0')]['response'].iloc[0][6]

    participant_sources['partisanship'] = int(partisanship_data)

    source_credibilities.loc[i] = participant_sources

    df.loc[df['participant'] == unique_participants[i], 'partisanship'] = int(partisanship_data)

In [ ]:
for column in ['CNN', 'Fox', 'NBC', 'ABC', 'CBS', 'USAtoday', 'NYT']:
    source_credibilities[column] = pd.to_numeric(source_credibilities[column])

In [ ]:
# Keep trials only, and recode responses to numerical
df = df.dropna(subset=['topic'])
df = df.reset_index(drop=True)
df['response'] = df['response'].replace({'f': -1, 'j': 1}).astype('category')

In [ ]:
# Partisanship as an ordered categorical variable
label_map = {
    0: 'Strongly Republican',
    1: 'Moderately Republican',
    2: 'Neutral',
    3: 'Moderately Democrat',
    4: 'Strongly Democrat'
}

df['partisanship'] = df['partisanship'].replace(label_map)

df['partisanship'] = pd.Categorical(df['partisanship'],
                                       categories=[
                                           'Strongly Republican',
                                           'Moderately Republican',
                                           'Neutral',
                                           'Moderately Democrat',
                                           'Strongly Democrat'
                                       ], ordered=True)

In [ ]:
# Participant exclusions: >25% responses under 1 s, or >25% timeouts
fast_responses = df.groupby('participant').apply(
    lambda x: (x['rt'] < 1000).mean() * 100
).reset_index(name='percent_fast_responses')

timeout_responses = df.groupby('participant').apply(
    lambda x: x['response'].isna().mean() * 100
).reset_index(name='percent_timeouts')

fast_excludes = fast_responses[fast_responses['percent_fast_responses'] > 25]
timeout_excludes = timeout_responses[timeout_responses['percent_timeouts'] > 25]

all_excludes = pd.concat([
    fast_excludes['participant'],
    timeout_excludes['participant']
]).unique()

df = df[~df['participant'].isin(all_excludes)]

In [ ]:
# Certainty: the prior for the topic of the trial, absolute and rescaled to 0-1
conditions = [
    df['topic'] == 'Climate Change',
    df['topic'] == 'Vaccination',
    df['topic'] == 'GMO',
    df['topic'] == '5G',
    df['topic'] == 'Voter Fraud'
]

choices = [
    df['prior_A'],
    df['prior_B'],
    df['prior_C'],
    df['prior_D'],
    df['prior_E']
]

df['certainty'] = np.select(conditions, choices, default=np.nan)
df['certainty'] = pd.to_numeric(df['certainty'])
df['certainty'] = df['certainty'].abs()
df['certainty'] = df['certainty']/100

In [ ]:
# Trial-level exclusions: responses under 1 s and neutral priors. Filler trials are left
# out of the denominator, and are removed by the dropna() in the next cell.
n_trials = (df['congruence'] != 'filler').sum()

df = df[(df['rt'] > 1000)]
n_fast = n_trials - (df['congruence'] != 'filler').sum()

# Removes nothing here: Experiment 2's congruence column has no 'Neutral' level
n_before_neutral = len(df)
df = df[df['congruence'] != 'Neutral']
n_neutral = n_before_neutral - len(df)

print(f"Belief-topic trials from retained participants: {n_trials:>6}")
print(f"  removed, fast responses (< 1 s):             {n_fast:>6}  ({n_fast / n_trials:.2%})")
print(f"  removed, neutral prior (certainty = 0):      {n_neutral:>6}  ({n_neutral / n_trials:.2%})")
print(f"  remaining:                                   {(df['congruence'] != 'filler').sum():>6}")
print(f"  zero-certainty trials still present:         {(df['certainty'] == 0).sum():>6}")

In [ ]:
# DDM dataframe: RTs in seconds, certainty negative on incongruent trials
df_ddm = df[['participant', 'rt', 'response', 'congruence', 'certainty', 'source_cat', 'source_num']]
df_ddm['rt'] = df_ddm['rt']/1000
df_ddm = df_ddm.dropna()

df_ddm.loc[df['congruence'] == 'Incongruent', 'certainty'] *= -1

df_ddm = df_ddm.rename(columns={'participant':'participant_id'})

df_ddm

## Set-up

In [ ]:
# Run this cell when skipping the cleaning steps above
df_ddm = pd.read_csv('df_study2_cleaned.csv')

In [ ]:
import arviz as az
import hssm
import hssm.plotting

## Descriptive statistics

Response coding: `1` = "true", `-1` = "false". RT in seconds.

In [ ]:
desc = df_ddm.copy()
desc["is_true"] = (desc["response"] == 1).astype(int)
desc["answer"] = np.where(desc["is_true"] == 1, '"true"', '"false"')

print(f"{len(desc)} trials from {desc['participant_id'].nunique()} participants")

### Proportion of "true" responses by congruence

In [ ]:
trial_level = desc.groupby("congruence", observed=True)["is_true"].agg(
    n_trials="size", n_true="sum", prop_true="mean"
)

by_pp = (
    desc.groupby(["participant_id", "congruence"], observed=True)["is_true"]
    .mean()
    .unstack("congruence")
)

n_pp = by_pp.notna().sum()
sem = by_pp.std() / np.sqrt(n_pp)
prop_true = trial_level.assign(
    n_participants=n_pp,
    pp_mean=by_pp.mean(),
    pp_sd=by_pp.std(),
    ci95_lo=by_pp.mean() - 1.96 * sem,
    ci95_hi=by_pp.mean() + 1.96 * sem,
)
display(prop_true.round(3))

# Within-participant congruent - incongruent difference
h1 = (by_pp["Congruent"] - by_pp["Incongruent"]).dropna()
h1_ci = 1.96 * h1.std() / np.sqrt(len(h1))
print('Within-participant difference in P("true"), congruent - incongruent:')
print(f"  mean = {h1.mean():.3f}, "
      f"95% CI [{h1.mean() - h1_ci:.3f}, {h1.mean() + h1_ci:.3f}], n = {len(h1)}")
print(f"  in the predicted direction for {(h1 > 0).sum()} of {len(h1)} participants "
      f"({(h1 > 0).mean():.1%})")

### Choice by congruence and certainty

In [ ]:
# certainty is signed in df_ddm, so take the magnitude back out
desc["certainty_mag"] = desc["certainty"].abs()

# Equal-width bands rather than terciles: about a third of trials sit at exactly 1.0
band_labels = ["low (0-0.33]", "medium (0.33-0.67]", "high (0.67-1.0]"]
desc["certainty_band"] = pd.cut(
    desc["certainty_mag"], bins=[0, 1 / 3, 2 / 3, 1.0],
    labels=band_labels, include_lowest=True,
)


def pp_summary(data, group_cols):
    """Participant-level mean of P("true") per cell, with a 95% CI across participants."""
    per_pp = (
        data.groupby(["participant_id"] + group_cols, observed=True)["is_true"]
        .mean()
        .rename("p_true")
        .reset_index()
    )
    out = per_pp.groupby(group_cols, observed=True)["p_true"].agg(
        n_participants="size", mean="mean", sd="std"
    )
    out["sem"] = out["sd"] / np.sqrt(out["n_participants"])
    out["ci95_lo"] = out["mean"] - 1.96 * out["sem"]
    out["ci95_hi"] = out["mean"] + 1.96 * out["sem"]
    out["n_trials"] = data.groupby(group_cols, observed=True).size()
    return out


band_summary = pp_summary(desc, ["certainty_band", "congruence"])
display(band_summary.round(3))

wide = band_summary["mean"].unstack("congruence")
print('Congruent - incongruent gap in P("true"), by certainty band:')
for band in band_labels:
    print(f"  {band:20} {wide.loc[band, 'Congruent'] - wide.loc[band, 'Incongruent']:.3f}")

### Choice by congruence and source credibility

In [ ]:
# source_cat is 'highest' / 'lowest': the participant's own highest- and lowest-rated
# news source, the same coding the DDM uses for z
cred_summary = pp_summary(desc, ["congruence", "source_cat"])
display(cred_summary.round(3))

per_pp = (
    desc.groupby(["participant_id", "congruence", "source_cat"], observed=True)["is_true"]
    .mean()
    .unstack("source_cat")
)
per_pp["cred_effect"] = per_pp["highest"] - per_pp["lowest"]


def mean_ci(values):
    values = values.dropna()
    return values.mean(), 1.96 * values.std() / np.sqrt(len(values)), len(values)


print('Credibility effect on P("true"), high - low, within participants:')
for cong in ["Congruent", "Incongruent"]:
    m, e, n = mean_ci(per_pp.xs(cong, level="congruence")["cred_effect"])
    print(f"  {cong:12} {m:+.3f}  95% CI [{m - e:+.3f}, {m + e:+.3f}]  n = {n}")

inter = per_pp["cred_effect"].unstack("congruence").dropna()
m, e, n = mean_ci(inter["Congruent"] - inter["Incongruent"])
print(f"  {'interaction':12} {m:+.3f}  95% CI [{m - e:+.3f}, {m + e:+.3f}]  n = {n}")

by_cong = (
    desc.groupby(["participant_id", "congruence"], observed=True)["is_true"]
    .mean()
    .unstack("congruence")
)
cong_effect, _, _ = mean_ci(by_cong["Congruent"] - by_cong["Incongruent"])
overall_cred, _, _ = mean_ci(per_pp["cred_effect"])
print(f"\nFor scale: congruence moves P(\"true\") by {cong_effect:.3f}, credibility by "
      f"{overall_cred:.3f} -- a factor of roughly {cong_effect / overall_cred:.0f}.")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.5))

# (left) P("true") by congruence and source credibility. Colour encodes congruence; the
# least-credible source keeps the same colour but is hatched.
congs = ["Congruent", "Incongruent"]
cong_colors = {"Congruent": "tab:blue", "Incongruent": "tab:red"}
x = np.arange(len(congs))
width = 0.38

for k, (src, hatch) in enumerate([("highest", ""), ("lowest", "///")]):
    means = [cred_summary.loc[(c, src), "mean"] for c in congs]
    errs = [1.96 * cred_summary.loc[(c, src), "sem"] for c in congs]
    pos = x + (k - 0.5) * width
    axL.bar(pos, means, width, yerr=errs, capsize=5,
            color=[cong_colors[c] for c in congs], alpha=0.85,
            hatch=hatch, edgecolor="white")
    for xi, val in zip(pos, means):
        axL.text(xi, val + 0.03, f"{val:.2f}", ha="center", fontsize=9)

axL.axhline(0.5, color="black", ls=":", lw=1)
axL.set_xticks(x)
axL.set_xticklabels(congs)
axL.set_ylim(0, 1)
axL.set_ylabel('P("true")')
axL.set_title('Proportion of "true" responses')

axL.legend(handles=[
    Patch(facecolor="grey", edgecolor="white", label="Most credible source"),
    Patch(facecolor="grey", edgecolor="white", hatch="///", label="Least credible source"),
    plt.Line2D([0], [0], color="black", ls=":", lw=1, label="chance"),
], loc="upper right")

# (right) P("true") across the certainty scale at 0.1 resolution, one line per congruence
bin_edges = np.round(np.arange(0, 1.01, 0.1), 2)
desc["certainty_bin"] = pd.cut(desc["certainty_mag"], bins=bin_edges, include_lowest=True)
fine_summary = pp_summary(desc, ["certainty_bin", "congruence"])
bin_cats = desc["certainty_bin"].cat.categories
centers = np.array([iv.mid for iv in bin_cats])

for cong, color in [("Congruent", "tab:blue"), ("Incongruent", "tab:red")]:
    sub = fine_summary.xs(cong, level="congruence").reindex(list(bin_cats))
    axR.plot(centers, sub["mean"], marker="o", markersize=4, lw=2, color=color, label=cong)
    axR.fill_between(centers, sub["ci95_lo"], sub["ci95_hi"], color=color, alpha=0.2)
axR.axhline(0.5, color="black", ls=":", lw=1, label="chance")
axR.set_xlabel("Certainty in prior belief")
axR.set_ylabel('P("true")')
axR.set_ylim(0, 1)
axR.set_title("Choice by congruence and certainty")
axR.legend(loc="center right")

plt.tight_layout()
plt.show()

### Response times by congruence

In [ ]:
rt_summary = desc.groupby(["congruence", "answer"], observed=True)["rt"].agg(
    n="size",
    mean="mean",
    sd="std",
    median="median",
    q25=lambda s: s.quantile(0.25),
    q75=lambda s: s.quantile(0.75),
)
display(rt_summary.round(2))

bins = np.linspace(desc["rt"].min(), desc["rt"].max(), 60)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax, cong in zip(axes, ["Congruent", "Incongruent"]):
    for ans, color in [('"true"', "tab:blue"), ('"false"', "tab:red")]:
        vals = desc.loc[(desc["congruence"] == cong) & (desc["answer"] == ans), "rt"]
        ax.hist(vals, bins=bins, density=True, alpha=0.5, color=color,
                label=f"{ans}  (Mdn = {vals.median():.2f} s)")
        ax.axvline(vals.median(), color=color, ls="--", lw=1.5)
    ax.set_title(f"{cong} headlines")
    ax.set_xlabel("RT (s)")
    ax.legend()
axes[0].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
# Collapsing across response
rt_by_cong = desc.groupby("congruence", observed=True)["rt"].agg(
    n="size",
    mean="mean",
    sd="std",
    median="median",
    q25=lambda s: s.quantile(0.25),
    q75=lambda s: s.quantile(0.75),
)
display(rt_by_cong.round(2))

print("Incongruent - congruent: "
      f"median {rt_by_cong.loc['Incongruent', 'median'] - rt_by_cong.loc['Congruent', 'median']:+.3f} s, "
      f"mean {rt_by_cong.loc['Incongruent', 'mean'] - rt_by_cong.loc['Congruent', 'mean']:+.3f} s")

bins = np.linspace(desc["rt"].min(), desc["rt"].max(), 60)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for cong, color in [("Congruent", "tab:blue"), ("Incongruent", "tab:red")]:
    vals = desc.loc[desc["congruence"] == cong, "rt"]
    ax.hist(vals, bins=bins, density=True, alpha=0.45, color=color,
            label=f"{cong}  (Mdn = {vals.median():.2f} s)")
    ax.axvline(vals.median(), color=color, ls="--", lw=1.5)
ax.set_xlabel("RT (s)")
ax.set_ylabel("Density")
ax.set_title("RT distribution by congruence")
ax.legend()

plt.tight_layout()
plt.show()

### Response times by certainty

In [ ]:
desc["prior_consistent"] = np.where(
    (desc["is_true"] == 1) == (desc["congruence"] == "Congruent"),
    "prior-consistent",
    "prior-inconsistent",
)


def pp_rt(data, group_cols):
    """Participant-level median RT per cell, summarised across participants."""
    per_pp = (
        data.groupby(["participant_id"] + group_cols, observed=True)["rt"]
        .median()
        .rename("median_rt")
        .reset_index()
    )
    out = per_pp.groupby(group_cols, observed=True)["median_rt"].agg(
        n_participants="size", mean="mean", sd="std"
    )
    out["sem"] = out["sd"] / np.sqrt(out["n_participants"])
    out["ci95_lo"] = out["mean"] - 1.96 * out["sem"]
    out["ci95_hi"] = out["mean"] + 1.96 * out["sem"]
    out["n_trials"] = data.groupby(group_cols, observed=True).size()
    return out


rt_by_cong = pp_rt(desc, ["certainty_band", "congruence"])
rt_by_cons = pp_rt(desc, ["certainty_band", "prior_consistent"])
display(rt_by_cong.round(2))
display(rt_by_cons.round(2))

for label, table, level, hi, lo in [
    ("congruence", rt_by_cong, "congruence", "Incongruent", "Congruent"),
    ("prior consistency", rt_by_cons, "prior_consistent",
     "prior-inconsistent", "prior-consistent"),
]:
    wide = table["mean"].unstack(level)
    print(f"Median RT difference by certainty band ({hi} - {lo}):")
    for band in band_labels:
        print(f"  {band:20} {wide.loc[band, hi] - wide.loc[band, lo]:+.3f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
x = np.arange(len(band_labels))

panels = [
    (axes[0], rt_by_cong, "congruence",
     [("Congruent", "tab:blue"), ("Incongruent", "tab:red")],
     "By congruence"),
    (axes[1], rt_by_cons, "prior_consistent",
     [("prior-consistent", "tab:purple"), ("prior-inconsistent", "tab:brown")],
     "By prior consistency of the response"),
]

for ax, table, level, series, title in panels:
    for name, color in series:
        sub = table.xs(name, level=level).reindex(band_labels)
        ax.errorbar(x, sub["mean"], yerr=1.96 * sub["sem"], marker="o", markersize=7,
                    capsize=5, lw=2, color=color, label=name)
    ax.set_xticks(x)
    ax.set_xticklabels(band_labels)
    ax.set_xlabel("Certainty in prior belief")
    ax.set_title(title)
    ax.legend(loc="upper left")

axes[0].set_ylabel("Median RT (s)")
fig.suptitle("Response time by certainty "
             "(points = participant mean of medians, 95% CI)")
plt.tight_layout()
plt.show()

### Overall RT distribution

In [ ]:
rt = desc["rt"]
print(f"RT (s): n = {len(rt)}, mean = {rt.mean():.2f}, SD = {rt.std():.2f}, "
      f"median = {rt.median():.2f}, "
      f"IQR = [{rt.quantile(0.25):.2f}, {rt.quantile(0.75):.2f}], "
      f"range = [{rt.min():.2f}, {rt.max():.2f}]")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(rt, bins=60, density=True, color="grey", alpha=0.55)
grid = np.linspace(rt.min(), rt.max(), 500)
ax.plot(grid, gaussian_kde(rt)(grid), color="black", lw=1.5)
ax.axvline(rt.median(), color="darkred", ls="--", lw=1.5,
           label=f"median = {rt.median():.2f} s")
ax.set_xlabel("RT (s)")
ax.set_ylabel("Density")
ax.set_title("RT distribution, all trials")
ax.legend()

plt.tight_layout()
plt.show()

## Functions

### ROPE analysis

In [ ]:
def rope_analysis(
    posterior,
    var_name,
    label=None,
    sd_multiplier=1,
    ci=0.95,
    interval="eti",
    title="Experiment 2",
    ax=None,
    verbose=True,
):
    """ROPE analysis for one parameter, with a ROPE of +-`sd_multiplier` posterior SDs
    around zero (Kruschke, 2013, 2018). `interval` is 'eti' or 'hdi'."""
    samples = posterior.posterior[var_name]

    # A categorical predictor adds a dimension holding its (single) non-reference level
    level = None
    for dim in [d for d in samples.dims if d not in ("chain", "draw")]:
        if samples.sizes[dim] != 1:
            raise ValueError(
                f"'{var_name}' has {samples.sizes[dim]} levels along '{dim}'; "
                "pick a single level before running a ROPE analysis."
            )
        if dim in samples.coords:
            level = str(samples.coords[dim].values[0])
    samples = samples.values.flatten()

    mean, sd = samples.mean(), samples.std()
    rope = sd_multiplier * sd

    if interval == "eti":
        tail = (1 - ci) / 2
        lower, upper = np.quantile(samples, (tail, 1 - tail))
    elif interval == "hdi":
        lower, upper = az.hdi(samples, hdi_prob=ci)
    else:
        raise ValueError("interval must be 'eti' or 'hdi'")

    inside = np.mean((samples > -rope) & (samples < rope))
    nearest = lower if abs(lower) < abs(upper) else upper

    if lower > rope or upper < -rope:
        decision = "entirely outside the ROPE -> reject the null value"
    elif -rope <= lower and upper <= rope:
        decision = "entirely inside the ROPE -> accept the null value"
    else:
        decision = "partially inside the ROPE -> remain undecided"

    if ax is None:
        _, ax = plt.subplots(1, figsize=(6, 4))

    kde = gaussian_kde(samples)
    grid = np.linspace(samples.min(), samples.max(), 1000)
    shaded = grid[(grid > lower) & (grid < upper)]

    ax.plot(grid, kde(grid), color="black")
    ax.fill_between(
        shaded, kde(shaded), color="darkred", alpha=0.25,
        label=f"{ci:.0%} {interval.upper()}",
    )
    ax.axvline(-rope, color="r", linestyle="--")
    ax.axvline(rope, color="r", linestyle="--", label=f"ROPE (\u00b1{sd_multiplier} SD)")
    ax.set_title(title)
    ax.set_xlabel(label if label is not None else var_name)
    ax.set_ylabel("Density")
    ax.legend()

    if verbose:
        name = f"{var_name}[{level}]" if level else var_name
        print(name)
        print(f"  posterior mean  = {mean:.3f}   SD = {sd:.3f}")
        print(f"  {ci:.0%} {interval.upper()}        = [{lower:.3f}, {upper:.3f}]")
        print(f"  ROPE            = [{-rope:.3f}, {rope:.3f}]")
        print(f"  bound nearest 0 = {nearest:.3f} "
              f"({abs(nearest) / sd:.2f} posterior SDs from zero)")
        print(f"  {inside:.1%} of the posterior falls inside the ROPE")
        print(f"  -> {decision}")

    return {
        "parameter": var_name,
        "level": level,
        "mean": mean,
        "sd": sd,
        "lower": lower,
        "upper": upper,
        "rope": (-rope, rope),
        "prop_in_rope": inside,
        "sds_from_zero": abs(nearest) / sd,
        "decision": decision,
    }

### Posterior predictive checks

In [ ]:
def plot_ppc(model, n_samples=20, col=None, **kwargs):
    """Posterior predictive check for a fitted model, plotted as signed RTs.

    `n_samples` is the number of posterior draws used, and is the step that costs time.
    `col` facets the check on a column of the data, e.g. "congruence".
    """
    # Renamed to plot_predictive after HSSM 0.2.x
    plot_fn = getattr(hssm.plotting, "plot_posterior_predictive", None)
    if plot_fn is None:
        plot_fn = hssm.plotting.plot_predictive

    return plot_fn(model, n_samples=n_samples, col=col, **kwargs)

### Parameter tables

In [ ]:
MAIN_PARAMS = ("v", "a", "z", "t")


def _param_order(name):
    """Sort key putting parameters in v, a, z, t order, each intercept first."""
    base = name.split("[")[0]
    param = base.split("_")[0]
    rank = MAIN_PARAMS.index(param) if param in MAIN_PARAMS else len(MAIN_PARAMS)
    is_effect = base not in (param, f"{param}_Intercept")
    return (rank, is_effect, base)


def main_summary(posterior, extra=(), **kwargs):
    """`az.summary` restricted to the population-level parameters of a DDM fit.

    Keeps the four DDM parameters and every effect the model puts on them, and drops
    the by-participant effects, their offsets and their group SDs. A categorical effect
    keeps its level in the row label and is a contrast against the reference level.
    """
    def is_population_level(name):
        if "|" in name:
            return False
        param = name.split("_")[0]
        if param not in MAIN_PARAMS:
            return False
        if name != param:
            return True
        # A formula-specified parameter is also stored under its bare name, as a
        # per-trial deterministic; keep the bare name only where it is one number.
        var = posterior.posterior[name]
        return not [dim for dim in var.dims if dim not in ("chain", "draw")]

    var_names = [name for name in posterior.posterior.data_vars
                 if is_population_level(name)]

    unknown = [name for name in extra if name not in posterior.posterior.data_vars]
    if unknown:
        print("not in this posterior:", ", ".join(unknown))
    var_names += [name for name in extra
                  if name not in var_names and name not in unknown]

    absent = [param for param in MAIN_PARAMS
              if not any(name == param or name.startswith(f"{param}_")
                         for name in var_names)]
    if absent:
        print("no population-level row found for:", ", ".join(absent))

    summary = az.summary(posterior, var_names=sorted(var_names, key=_param_order),
                         **kwargs)
    return summary.loc[sorted(summary.index, key=_param_order)]

## Models

Sampling passes `initvals={"t": 0.5}`: the default of 2.0 s is above the fastest RTs in
this dataset, which sends every draw divergent.

### Model 1

In [ ]:
ddm_model_1 = hssm.HSSM(
    data=df_ddm,
    prior_settings='safe',
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": -2.0, "upper": 2.0},
                "congruence": {"name": "Uniform", "lower": -4.0, "upper": 1.0},
            },
            "formula": "v ~ 1 + congruence + (congruence|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
            },
            "formula": "a ~ 1 + (1|participant_id)"
        },
    ],)

In [ ]:
posterior_ddm_model_1 = ddm_model_1.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_1)

In [ ]:
az.summary(posterior_ddm_model_1)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_1)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_1)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_1, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
]:
    rope_analysis(posterior_ddm_model_1, var, label=label)
    plt.show()

### Model 2

In [ ]:
ddm_model_2 = hssm.HSSM(
    data=df_ddm,
    prior_settings='safe',
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 2.5},
                "congruence": {"name": "Uniform", "lower": -3.5, "upper": 0},
            },
            "formula": "v ~ 1 + congruence + (congruence|participant_id)",
            "link": "identity",
        },
       {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
                "congruence": {"name": "Uniform", "lower": -2.0, "upper": 2.0},
            },
            "formula": "a ~ 1 + congruence + (congruence|participant_id)"
        },
    ],)

In [ ]:
posterior_ddm_model_2 = ddm_model_2.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_2)

In [ ]:
az.summary(posterior_ddm_model_2)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_2)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_2)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_2, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("a_congruence", "Effect of incongruence on boundary separation (a)"),
]:
    rope_analysis(posterior_ddm_model_2, var, label=label)
    plt.show()

### Model 3

In [ ]:
ddm_model_3 = hssm.HSSM(
    data=df_ddm,
    prior_settings='safe',
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": -2.0, "upper": 2.0},
                "congruence": {"name": "Uniform", "lower": -5.0, "upper": 5.0},
                "certainty": {"name": "Uniform", "lower": -5, "upper": 5},
            },
            "formula": "v ~ 1 + congruence + certainty + (congruence|participant_id) + (certainty|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
            },
            "formula": "a ~ 1 + (1|participant_id)"
        },
    ],)

In [ ]:
posterior_ddm_model_3 = ddm_model_3.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_3)

In [ ]:
az.summary(posterior_ddm_model_3)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_3)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_3)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_3, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("v_certainty", "Effect of certainty on drift rate (v)"),
]:
    rope_analysis(posterior_ddm_model_3, var, label=label)
    plt.show()

### Model 4

In [ ]:
ddm_model_4 = hssm.HSSM(
    data=df_ddm,
    prior_settings="safe",
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 3},
                "congruence": {"name": "Uniform", "lower": -3.0, "upper": 0},
                "certainty": {"name": "Uniform", "lower": 0, "upper": 3},
            },
            "formula": "v ~ congruence + certainty + (congruence|participant_id) + (certainty|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
                "congruence": {"name": "Uniform", "lower": -2.0, "upper": 2.0},
            },
            "formula": "a ~ 1 + congruence + (congruence|participant_id)",
            "link": "identity",
        },
    ],)

In [ ]:
posterior_ddm_model_4 = ddm_model_4.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_4)

In [ ]:
az.summary(posterior_ddm_model_4)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_4)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_4)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_4, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("v_certainty", "Effect of certainty on drift rate (v)"),
    ("a_congruence", "Effect of incongruence on boundary separation (a)"),
]:
    rope_analysis(posterior_ddm_model_4, var, label=label)
    plt.show()

### Model 5

In [ ]:
ddm_model_5 = hssm.HSSM(
    data=df_ddm,
    prior_settings='safe',
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 2.5},
                "congruence": {"name": "Uniform", "lower": -3.5, "upper": 0},
            },
            "formula": "v ~ 1 + congruence + (congruence|participant_id)",
            "link": "identity",
        },
        {
            "name": "z",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.5, "sigma": 0.01},
                "source_cat": {"name": "Uniform", "lower": -1, "upper": 0},
            },
            "formula": "z ~ 1 + source_cat + (source_cat|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
            },
            "formula": "a ~ 1 + (1|participant_id)"
        },
    ],)

In [ ]:
posterior_ddm_model_5 = ddm_model_5.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_5)

In [ ]:
az.summary(posterior_ddm_model_5)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_5)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_5)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_5, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("z_source_cat", "Effect of low perceived source credibility on bias (z)"),
]:
    rope_analysis(posterior_ddm_model_5, var, label=label)
    plt.show()

### Model 6

In [ ]:
ddm_model_6 = hssm.HSSM(
    data=df_ddm,
    prior_settings="safe",
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 3},
                "congruence": {"name": "Uniform", "lower": -3.0, "upper": 0},
                "certainty": {"name": "Uniform", "lower": 0, "upper": 3},
            },
            "formula": "v ~ congruence + certainty + (congruence|participant_id) + (certainty|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
            },
            "formula": "a ~ 1 + (1|participant_id)",
            "link": "identity",
        },
        {
            "name": "z",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.5, "sigma": 0.01},
                "source_cat": {"name": "Uniform", "lower": -1, "upper": 1},
            },
            "formula": "z ~ 1 + source_cat + (source_cat|participant_id)",
            "link": "identity",
        },
    ],)

In [ ]:
posterior_ddm_model_6 = ddm_model_6.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_6)

In [ ]:
az.summary(posterior_ddm_model_6)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_6)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_6)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_6, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("v_certainty", "Effect of certainty on drift rate (v)"),
    ("z_source_cat", "Effect of low perceived source credibility on bias (z)"),
]:
    rope_analysis(posterior_ddm_model_6, var, label=label)
    plt.show()

### Model 7 (exploratory)

Model 6 with congruence and source credibility on boundary separation. `source_cat` is on
both `a` and `z`.

In [ ]:
ddm_model_7 = hssm.HSSM(
    data=df_ddm,
    prior_settings="safe",
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 3},
                "congruence": {"name": "Uniform", "lower": -3.0, "upper": 0},
                "certainty": {"name": "Uniform", "lower": 0, "upper": 3},
            },
            "formula": "v ~ congruence + certainty + (congruence|participant_id) + (certainty|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
                "congruence": {"name": "Uniform", "lower": -2.0, "upper": 2.0},
                "source_cat": {"name": "Uniform", "lower": -1.0, "upper": 1.0},
            },
            "formula": "a ~ 1 + congruence + source_cat + (congruence + source_cat | participant_id)",
            "link": "identity",
        },
        {
            "name": "z",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.5, "sigma": 0.01},
                "source_cat": {"name": "Uniform", "lower": -1, "upper": 0},
            },
            "formula": "z ~ 1 + source_cat + (source_cat|participant_id)",
            "link": "identity",
        },
    ],)

In [ ]:
posterior_ddm_model_7 = ddm_model_7.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_7)

In [ ]:
az.summary(posterior_ddm_model_7)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_7)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_7)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_7, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("v_certainty", "Effect of certainty on drift rate (v)"),
    ("a_congruence", "Effect of incongruence on boundary separation (a)"),
    ("a_source_cat", "Effect of low perceived source credibility on boundary separation (a)"),
    ("z_source_cat", "Effect of low perceived source credibility on bias (z)"),
]:
    rope_analysis(posterior_ddm_model_7, var, label=label)
    plt.show()

### Model 8 (exploratory)

Model 6 with belief strength (`cert_mag`, the magnitude of `certainty`) on boundary
separation.

In [ ]:
# Belief strength, independent of direction
df_ddm["cert_mag"] = df_ddm["certainty"].abs()

ddm_model_8 = hssm.HSSM(
    data=df_ddm,
    prior_settings="safe",
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 3},
                "congruence": {"name": "Uniform", "lower": -3.0, "upper": 0},
                "certainty": {"name": "Uniform", "lower": 0, "upper": 3},
            },
            "formula": "v ~ congruence + certainty + (congruence|participant_id) + (certainty|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
                "cert_mag": {"name": "Normal", "mu": 0.0, "sigma": 0.3},
            },
            "formula": "a ~ 1 + cert_mag + (cert_mag|participant_id)",
            "link": "identity",
        },
        {
            "name": "z",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.5, "sigma": 0.01},
                "source_cat": {"name": "Uniform", "lower": -1, "upper": 0},
            },
            "formula": "z ~ 1 + source_cat + (source_cat|participant_id)",
            "link": "identity",
        },
    ],)

In [ ]:
posterior_ddm_model_8 = ddm_model_8.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_8)

In [ ]:
az.summary(posterior_ddm_model_8)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_8)
plt.tight_layout()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_8)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_8, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("v_certainty", "Effect of certainty on drift rate (v)"),
    ("a_cert_mag", "Effect of belief strength on boundary separation (a)"),
    ("z_source_cat", "Effect of low perceived source credibility on bias (z)"),
]:
    rope_analysis(posterior_ddm_model_8, var, label=label)
    plt.show()

### Model 9 (exploratory)

Model 6 with source credibility on drift rate. `source_cat` is on both `v` and `z`.

In [ ]:
ddm_model_9 = hssm.HSSM(
    data=df_ddm,
    prior_settings="safe",
    include=[
        {
            "name": "v",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 0, "upper": 3},
                "congruence": {"name": "Uniform", "lower": -3.0, "upper": 0},
                "certainty": {"name": "Uniform", "lower": 0, "upper": 3},
                "source_cat": {"name": "Normal", "mu": 0.0, "sigma": 0.3},
            },
            "formula": "v ~ congruence + certainty + source_cat + (congruence|participant_id) + (certainty|participant_id) + (source_cat|participant_id)",
            "link": "identity",
        },
        {
            "name": "a",
            "prior": {
                "Intercept": {"name": "Uniform", "lower": 1, "upper": 3.5},
            },
            "formula": "a ~ 1 + (1|participant_id)",
            "link": "identity",
        },
        {
            "name": "z",
            "prior": {
                "Intercept": {"name": "Normal", "mu": 0.5, "sigma": 0.01},
                "source_cat": {"name": "Uniform", "lower": -1, "upper": 0},
            },
            "formula": "z ~ 1 + source_cat + (source_cat|participant_id)",
            "link": "identity",
        },
    ],)

In [ ]:
posterior_ddm_model_9 = ddm_model_9.sample(
    sampler="nuts_numpyro",
    cores=3,
    chains=3,
    draws=2000,
    tune=3000,
    initvals={"t": 0.5},
    idata_kwargs=dict(log_likelihood=True),
)

#### Parameter estimates

In [ ]:
main_summary(posterior_ddm_model_9)

In [ ]:
az.summary(posterior_ddm_model_9)

#### Trace plots

In [ ]:
az.plot_trace(posterior_ddm_model_9)
plt.tight_layout()

#### Drift / starting-point trade-off

In [ ]:
# source_cat sits on both v and z here, so check whether the two terms trade off
vs = posterior_ddm_model_9.posterior["v_source_cat"].values.ravel()
zs = posterior_ddm_model_9.posterior["z_source_cat"].values.ravel()
print(f"corr(v_source_cat, z_source_cat) = {np.corrcoef(vs, zs)[0, 1]:+.3f}")

az.plot_pair(
    posterior_ddm_model_9,
    var_names=["v_source_cat", "z_source_cat"],
    kind="kde",
)
plt.show()

#### Posterior predictive check

In [ ]:
plot_ppc(ddm_model_9)
plt.show()

# Faceted by condition
# plot_ppc(ddm_model_9, col="congruence")
# plt.show()

#### ROPE analysis

In [ ]:
for var, label in [
    ("v_congruence", "Effect of incongruence on drift rate (v)"),
    ("v_certainty", "Effect of certainty on drift rate (v)"),
    ("v_source_cat", "Effect of low perceived source credibility on drift rate (v)"),
    ("z_source_cat", "Effect of low perceived source credibility on bias (z)"),
]:
    rope_analysis(posterior_ddm_model_9, var, label=label)
    plt.show()

## Model comparison

ELPD is computed per model with `az.waic`, from that model's own log-likelihood.

In [ ]:
posteriors = {
    1: posterior_ddm_model_1,
    2: posterior_ddm_model_2,
    3: posterior_ddm_model_3,
    4: posterior_ddm_model_4,
    5: posterior_ddm_model_5,
    6: posterior_ddm_model_6,
    7: posterior_ddm_model_7,
    8: posterior_ddm_model_8,
    9: posterior_ddm_model_9,
}


def rank_models(df):
    """Order best to worst and recompute the difference from the best model.

    Each model's elpd_waic comes from its own log-likelihood, so a subset can be
    re-ranked without recomputing anything.
    """
    out = df.sort_values("elpd_waic", ascending=False).copy()
    out["rank"] = range(len(out))
    out["elpd_diff"] = out["elpd_waic"].iloc[0] - out["elpd_waic"]
    return out


def waic_frame(model_numbers):
    """ELPD per model, in the column layout `compare_to_apa` expects."""
    rows = {}
    for n in model_numbers:
        w = az.waic(posteriors[n], scale="log")
        rows[f"Model {n}"] = {"elpd_waic": float(w.elpd_waic),
                              "p_waic": float(w.p_waic),
                              "se": float(w.se)}
        print(f"Model {n}: elpd_waic = {w.elpd_waic:.2f}")
    return rank_models(pd.DataFrame(rows).T)


waic_all = waic_frame(range(1, 10))

# Preregistered models, and the full set including the exploratory models
compare_data_prereg = rank_models(waic_all.loc[[f"Model {n}" for n in range(1, 7)]])
compare_data_all = waic_all

compare_data_all

### APA tables

Writes `<filename>.html`, which Word opens as a native table, and `<filename>.docx` if
`python-docx` is installed.

In [ ]:
import re

NUMBER_WORDS = {2: "two", 3: "three", 4: "four", 5: "five", 6: "six",
                7: "seven", 8: "eight", 9: "nine", 10: "ten"}


def compare_to_apa(compare_df, table_number, title, note=None, filename=None):
    """Render an `az.compare` frame as an APA table of ranked ELPD values."""
    ic = "waic" if "elpd_waic" in compare_df.columns else "loo"
    df = compare_df.sort_values("rank")

    # az.compare is keyed on the labels handed to it ("Model 4"); the tables carry
    # the bare number
    numbers = [re.sub(r"^\D*", "", str(i)) or str(i) for i in df.index]
    elpd = [f"{v:.2f}" for v in df[f"elpd_{ic}"]]

    cols = [
        ("Model Number", numbers),
        (f"ELPD ({ic.upper()})", elpd),
    ]
    table = pd.DataFrame({label: values for label, values in cols})

    if note is None:
        count = NUMBER_WORDS.get(len(df), str(len(df)))
        note = (
            f"Expected Log Pointwise Predictive Density (ELPD) for each of the "
            f"{count} models. Higher values indicate better model fit. Models are "
            f"ordered from best to worst fit."
        )

    if filename is not None:
        with open(f"{filename}.html", "w", encoding="utf-8") as f:
            f.write(_apa_html(cols, table_number, title, note))
        print(f"wrote {filename}.html")
        try:
            _apa_docx(cols, table_number, title, note, f"{filename}.docx")
            print(f"wrote {filename}.docx")
        except ImportError:
            print("      (pip install python-docx to also get a native .docx)")

    return table


def _apa_html(cols, table_number, title, note):
    """APA table as standalone HTML: horizontal rules only, no vertical lines."""
    rule = "1px solid #000"
    n_rows = len(cols[0][1])

    head = "".join(
        f'<th style="border-top:{rule};border-bottom:{rule};padding:8px 24px;'
        f'text-align:center;">{label}</th>'
        for label, _ in cols
    )
    rows = ""
    for r in range(n_rows):
        edge = f"border-bottom:{rule};" if r == n_rows - 1 else ""
        rows += "<tr>" + "".join(
            f'<td style="{edge}padding:8px 24px;text-align:center;">{values[r]}</td>'
            for _, values in cols
        ) + "</tr>"

    return f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Table {table_number}</title></head>
<body style="font-family:'Times New Roman',serif;font-size:12pt;color:#000;">
<p style="margin:0;"><b>Table {table_number}</b></p>
<p style="margin:0 0 8pt 0;"><i>{title}</i></p>
<table style="border-collapse:collapse;">
<thead><tr>{head}</tr></thead>
<tbody>{rows}</tbody>
</table>
<p style="margin:10pt 0 0 0;"><i>Note.</i> {note}</p>
</body></html>"""


def _apa_docx(cols, table_number, title, note, path):
    """The same table as a native Word file. Raises ImportError without python-docx."""
    from docx import Document
    from docx.shared import Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

    def rule(cell, edge):
        tcPr = cell._tc.get_or_add_tcPr()
        borders = tcPr.find(qn("w:tcBorders"))
        if borders is None:
            borders = OxmlElement("w:tcBorders")
            tcPr.append(borders)
        el = OxmlElement(f"w:{edge}")
        el.set(qn("w:val"), "single")
        el.set(qn("w:sz"), "6")
        el.set(qn("w:color"), "000000")
        borders.append(el)

    doc = Document()
    doc.styles["Normal"].font.name = "Times New Roman"
    doc.styles["Normal"].font.size = Pt(12)

    doc.add_paragraph().add_run(f"Table {table_number}").bold = True
    doc.add_paragraph().add_run(title).italic = True

    n_rows = len(cols[0][1])
    tbl = doc.add_table(rows=n_rows + 1, cols=len(cols))
    for c, (label, values) in enumerate(cols):
        header = tbl.cell(0, c)
        header.paragraphs[0].add_run(label).bold = True
        rule(header, "top")
        rule(header, "bottom")
        for r, value in enumerate(values, start=1):
            body = tbl.cell(r, c)
            body.text = value
            if r == n_rows:
                rule(body, "bottom")
        for r in range(n_rows + 1):
            tbl.cell(r, c).paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER

    p = doc.add_paragraph()
    p.add_run("Note.").italic = True
    p.add_run(" " + note)
    doc.save(path)

In [ ]:
display(compare_to_apa(
    compare_data_prereg,
    table_number=4,
    title="Ranked Model Comparison by ELPD for Experiment 2",
    filename="Table4_model_comparison",
))

display(compare_to_apa(
    compare_data_all,
    table_number=5,
    title="Ranked Model Comparison by ELPD for Experiment 2, Including the Exploratory Models",
    filename="Table5_model_comparison",
))

## Parameter recovery

The estimates of the best-fitting model (Model 6) are taken as ground truth, 10 datasets
with the design of the experiment are simulated from them, and the model is refit to each.
Simulating and refitting is slow.

### True parameters

In [ ]:
RECOVERY_POSTERIOR = posterior_ddm_model_6
N_SIMULATIONS = 10
N_TOPICS = 5
N_TRIALS = 60
N_SUBJECTS = df_ddm["participant_id"].nunique()


def get_param(values, *names):
    """Look a parameter up under any of several names.

    `congruence` is categorical in df_ddm, so its coefficient is
    'v_congruence[Incongruent]'; the simulated data codes congruence as 0/1, which HSSM
    treats as continuous, so the refitted coefficient comes back as plain 'v_congruence'.
    """
    for name in names:
        if name in values.index:
            return values[name]
    raise KeyError(f"none of {names} found")


# Fixed effects, and the random-effect name patterns ({s} is filled with mu / sigma)
PARAM_NAMES = {
    "v_intercept": ("v_Intercept",),
    "v_incongruence": ("v_congruence[Incongruent]", "v_congruence"),
    "v_certainty": ("v_certainty",),
    "a_intercept": ("a_Intercept",),
    "a_incongruence": ("a_congruence[Incongruent]", "a_congruence"),
    "z_intercept": ("z_Intercept",),
    "z_low_cred": ("z_source_cat[lowest]",),
    "t": ("t",),
}

RE_NAMES = {
    "v_intercept": ("v_1|participant_id_{s}",),
    "v_incongruence": ("v_congruence|participant_id_{s}[Incongruent]",
                       "v_congruence|participant_id_{s}"),
    "v_certainty": ("v_certainty|participant_id_{s}",),
    "a_intercept": ("a_1|participant_id_{s}",),
    "a_incongruence": ("a_congruence|participant_id_{s}[Incongruent]",
                       "a_congruence|participant_id_{s}"),
    "z_intercept": ("z_1|participant_id_{s}",),
    "z_low_cred": ("z_source_cat|participant_id_{s}[lowest]",),
}

# 'a' is a trial-level deterministic, not a parameter to recover
param_values = az.summary(RECOVERY_POSTERIOR, var_names=["~a"])["mean"]

true_params, missing = {}, []
for key, names in PARAM_NAMES.items():
    try:
        true_params[key] = get_param(param_values, *names)
    except KeyError:
        missing.append(key)

for key, patterns in RE_NAMES.items():
    for stat in ("mu", "sigma"):
        try:
            true_params[f"{stat}_{key}"] = get_param(
                param_values, *(p.format(s=stat) for p in patterns))
        except KeyError:
            pass

# Participant-level effects the simulation draws: those with both a mu and a sigma
RANDOM_EFFECTS = [k for k in RE_NAMES
                  if f"mu_{k}" in true_params and f"sigma_{k}" in true_params]

certainty_kde = stats.gaussian_kde(np.abs(df_ddm["certainty"].values))
highest_kde = stats.gaussian_kde(
    df_ddm.loc[df_ddm["source_cat"] == "highest", "source_num"].values)
lowest_kde = stats.gaussian_kde(
    df_ddm.loc[df_ddm["source_cat"] == "lowest", "source_num"].values)

print(f"{N_SUBJECTS} participants x {N_TRIALS} trials per simulated dataset")
print("recovering:      ", ", ".join(k for k in PARAM_NAMES if k in true_params))
print("random effects:  ", ", ".join(RANDOM_EFFECTS))
if missing:
    print("not in this model:", ", ".join(missing))

### Simulation

In [ ]:
def simulate_recovery_dataset(true_params, n_subjects=None, n_trials=None, seed=None):
    """Simulate one dataset with the design of the experiment.

    Each participant gets `n_trials` trials split evenly across 5 topics; congruence is
    balanced within topic and source category within congruence. Certainty is drawn once
    per topic from the empirical KDE and made negative on incongruent trials, matching
    how it is coded in df_ddm. Each participant gets one credibility value for their
    highest- and one for their lowest-rated source, reused across their trials.
    """
    n_subjects = N_SUBJECTS if n_subjects is None else n_subjects
    n_trials = N_TRIALS if n_trials is None else n_trials
    rng = np.random.default_rng(seed)
    per_topic = n_trials // N_TOPICS

    rows, subject_random_effects = [], {}

    for subject in range(n_subjects):
        effects = {name: rng.normal(true_params[f"mu_{name}"], true_params[f"sigma_{name}"])
                   for name in RANDOM_EFFECTS}
        subject_random_effects[subject] = effects

        source_value = {"highest": highest_kde.resample(size=1)[0][0],
                        "lowest": lowest_kde.resample(size=1)[0][0]}

        topics = np.repeat(np.arange(N_TOPICS), per_topic)

        incongruence, source_cat = [], []
        for _ in range(N_TOPICS):
            half = per_topic // 2
            block_inc = np.concatenate([np.zeros(half), np.ones(per_topic - half)])
            block_src = np.concatenate([
                rng.permutation(["highest"] * (half // 2) + ["lowest"] * (half - half // 2)),
                rng.permutation(["highest"] * ((per_topic - half) // 2)
                                + ["lowest"] * (per_topic - half - (per_topic - half) // 2)),
            ])
            keep = rng.permutation(per_topic)
            incongruence.append(block_inc[keep])
            source_cat.append(block_src[keep])
        incongruence = np.concatenate(incongruence)
        source_cat = np.concatenate(source_cat)

        order = rng.permutation(len(topics))
        topics, incongruence, source_cat = topics[order], incongruence[order], source_cat[order]

        topic_certainty = certainty_kde.resample(size=N_TOPICS)[0]
        certainty = np.array([topic_certainty[int(t)] for t in topics]) * (1 - 2 * incongruence)

        for trial in range(len(topics)):
            v = (true_params["v_intercept"] + effects["v_intercept"]
                 + (true_params["v_incongruence"] + effects["v_incongruence"]) * incongruence[trial]
                 + (true_params["v_certainty"] + effects["v_certainty"]) * certainty[trial])

            a = true_params["a_intercept"] + effects["a_intercept"]
            if "a_incongruence" in true_params:
                a += (true_params["a_incongruence"]
                      + effects.get("a_incongruence", 0.0)) * incongruence[trial]

            z = true_params["z_intercept"] + effects["z_intercept"]
            if source_cat[trial] == "lowest":
                z += true_params["z_low_cred"] + effects["z_low_cred"]

            trial_data = hssm.simulate_data(
                model="ddm",
                theta=dict(v=v, a=a, z=z, t=true_params["t"]),
                size=1,
            )
            rows.append({
                "participant_id": subject,
                "trial": trial,
                "rt": trial_data["rt"][0],
                "response": trial_data["response"][0],
                "congruence": incongruence[trial],
                "certainty": certainty[trial],
                "topic": topics[trial],
                "source_cat": source_cat[trial],
                "source_num": source_value[source_cat[trial]],
            })

    return pd.DataFrame(rows), subject_random_effects

### Simulate and refit

In [ ]:
# Weakly informative priors, so the model's Uniform priors do not bound the recovered values
def fit_simulated(df_sim):
    ddm_model_sim = hssm.HSSM(
        data=df_sim,
        prior_settings="safe",
        include=[
            {
                "name": "v",
                "prior": {"Intercept": {"name": "Normal", "mu": 0.0, "sigma": 0.5}},
                "formula": "v ~ congruence + certainty + (congruence|participant_id) + (certainty|participant_id)",
                "link": "identity",
            },
            {
                "name": "a",
                "prior": {"Intercept": {"name": "Normal", "mu": 0.0, "sigma": 0.5}},
                "formula": "a ~ 1 + congruence + (congruence|participant_id)",
                "link": "identity",
            },
            {
                "name": "z",
                "prior": {"Intercept": {"name": "Normal", "mu": 0.0, "sigma": 0.5}},
                "formula": "z ~ 1 + source_cat + (source_cat|participant_id)",
                "link": "identity",
            },
        ],)

    return ddm_model_sim.sample(
        sampler="nuts_numpyro",
        cores=3,
        chains=3,
        draws=2000,
        tune=3000,
        idata_kwargs=dict(log_likelihood=True),
    )


sim_summaries = {}

for sim_nr in range(1, N_SIMULATIONS + 1):
    df_sim, subject_random_effects = simulate_recovery_dataset(true_params, seed=sim_nr)
    posterior_sim = fit_simulated(df_sim)
    sim_summaries[sim_nr] = az.summary(posterior_sim, var_names=["~a"])
    print(f"simulation {sim_nr}: {len(df_sim)} trials from "
          f"{df_sim['participant_id'].nunique()} participants")

In [ ]:
# Simulated against observed RTs, for the last simulation
fig, ax = plt.subplots(figsize=(7.5, 4.5))
bins = np.linspace(0, max(df_ddm["rt"].max(), df_sim["rt"].abs().max()), 60)
ax.hist(df_ddm["rt"], bins=bins, density=True, alpha=0.45, color="tab:blue",
        label=f"Experimental (Mdn = {df_ddm['rt'].median():.2f} s)")
ax.hist(df_sim["rt"].abs(), bins=bins, density=True, alpha=0.45, color="tab:red",
        label=f"Simulated (Mdn = {df_sim['rt'].abs().median():.2f} s)")
ax.set_xlabel("Reaction time (s)")
ax.set_ylabel("Density")
ax.set_title(f"Observed vs simulated RTs (simulation {sim_nr})")
ax.legend()
plt.tight_layout()
plt.show()

### Recovery metrics

In [ ]:
RECOVERED = [k for k in PARAM_NAMES if k in true_params]

# Parameters checked against the same +-1 SD ROPE
ROPE_PARAMS = [p for p in ["v_certainty", "a_incongruence", "z_low_cred", "v_incongruence"] if p in true_params]


def rope_status(sd, hdi_lower, hdi_upper):
    """Where the HDI falls relative to a ROPE of +-1 posterior SD around zero."""
    if hdi_lower > sd or hdi_upper < -sd:
        return "Outside"
    if hdi_lower >= -sd and hdi_upper <= sd:
        return "Inside"
    return "Partial"


rows = []
rope_counts = {p: {"Inside": 0, "Partial": 0, "Outside": 0} for p in ROPE_PARAMS}

for i, summary in sim_summaries.items():
    row = {"Simulation": i}
    for key in RECOVERED:
        names = PARAM_NAMES[key]
        row[key] = get_param(summary["mean"], *names)
        row[f"{key}_HDI_lower"] = get_param(summary["hdi_3%"], *names)
        row[f"{key}_HDI_upper"] = get_param(summary["hdi_97%"], *names)

    for p in ROPE_PARAMS:
        names = PARAM_NAMES[p]
        status = rope_status(get_param(summary["sd"], *names),
                             row[f"{p}_HDI_lower"], row[f"{p}_HDI_upper"])
        row[f"{p}_ROPE_status"] = status
        rope_counts[p][status] += 1

    rows.append(row)

params_df = pd.DataFrame(rows)

metrics = []
for key in RECOVERED:
    true_value = true_params[key]
    recovered = params_df[key].astype(float)
    bias = recovered.mean() - true_value
    covered = ((params_df[f"{key}_HDI_lower"] <= true_value)
               & (params_df[f"{key}_HDI_upper"] >= true_value))
    metrics.append({
        "True value": true_value,
        "Mean recovered": recovered.mean(),
        "SD recovered": recovered.std(),
        "Bias": bias,
        "Relative bias (%)": np.nan if true_value == 0 else bias / true_value * 100,
        "MAE": (recovered - true_value).abs().mean(),
        "RMSE": np.sqrt(((recovered - true_value) ** 2).mean()),
        "HDI coverage (%)": covered.mean() * 100,
    })

metrics_df = pd.DataFrame(metrics, index=RECOVERED)
metrics_df.index.name = "Parameter"
display(metrics_df.round(3))

print(f"\nROPE verdicts across {len(params_df)} simulations:")
for p, counts in rope_counts.items():
    print(f"  {p:16} " + ", ".join(f"{k}: {v}" for k, v in counts.items()))

params_df.to_csv("parameter_recovery_results_exp2.csv", index=False)
metrics_df.to_csv("parameter_recovery_metrics_exp2.csv")
print("\nwritten to parameter_recovery_results_exp2.csv "
      "and parameter_recovery_metrics_exp2.csv")

In [ ]:
plot_params = [p for p in ROPE_PARAMS if p in params_df]

fig, axes = plt.subplots(1, len(plot_params), figsize=(4.2 * len(plot_params), 4))
axes = np.atleast_1d(axes)

for ax, p in zip(axes, plot_params):
    recovered = params_df[p].astype(float)
    ax.hist(recovered, bins=10, color="tab:blue", alpha=0.6)
    ax.axvline(true_params[p], color="darkred", ls="--", lw=2,
               label=f"true = {true_params[p]:.3f}")
    ax.axvline(recovered.mean(), color="black", lw=1.5,
               label=f"recovered = {recovered.mean():.3f}")
    ax.set_title(p)
    ax.set_xlabel("Recovered value")
    ax.set_ylabel("Simulations")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()